# 1 - Функции для создания синтетической CoT для сложения и вычитания

In [ ]:
def generate_addition_cot(a: int, b: int) -> str:
    """Генерирует CoT-ответ для сложения двух чисел до 5 знаков."""
    assert 0 <= a <= 99999 and 0 <= b <= 99999, "Числа должны быть от 0 до 99999"
    
    str_a, str_b = str(a).zfill(5), str(b).zfill(5)
    carry = 0
    tokens = []
    
    # Итерация от младших разрядов к старшим
    for i in range(4, -1, -1):
        current_sum = int(str_a[i]) + int(str_b[i]) + carry
        carry = current_sum // 10
        digit = current_sum % 10
        
        tokens.extend([f"<c{carry}>", str(digit)])
        
    # Обработка 6-го разряда (для случаев переполнения, например 99999 + 1)
    # По твоему шаблону он присутствует всегда
    tokens.extend(["<c0>", str(carry)])
    
    return " ".join(tokens)


def generate_subtraction_cot(a: int, b: int) -> str:
    """Генерирует CoT-ответ для вычитания (a - b). Предполагается a >= b."""
    assert 0 <= b <= a <= 99999, "Должно выполняться 0 <= b <= a <= 99999"
    
    str_a, str_b = str(a).zfill(5), str(b).zfill(5)
    borrow = 0
    tokens = []
    
    # Итерация от младших разрядов к старшим
    for i in range(4, -1, -1):
        current_diff = int(str_a[i]) - int(str_b[i]) - borrow
        
        if current_diff < 0:
            current_diff += 10
            borrow = 1
        else:
            borrow = 0
            
        tokens.extend([f"<b{borrow}>", str(current_diff)])
        
    return " ".join(tokens)

In [3]:
# Тест сложения 
add_result = generate_addition_cot(45, 92)
print("Сложение:", add_result)

# Тест вычитания 
sub_result = generate_subtraction_cot(502, 19)
print("Вычитание:", sub_result)

assert add_result == "<c0> 7 <c1> 3 <c0> 1 <c0> 0 <c0> 0 <c0> 0"
assert sub_result == "<b1> 3 <b1> 8 <b0> 4 <b0> 0 <b0> 0"

Сложение: <c0> 7 <c1> 3 <c0> 1 <c0> 0 <c0> 0 <c0> 0
Вычитание: <b1> 3 <b1> 8 <b0> 4 <b0> 0 <b0> 0


# 2 - Случайная(с учётом разницы в количестве чисел разной длины + без повторений) генерация отдельных пулов пар чисел для сложения и вычитания

In [5]:
import random

def get_random_num(length: int) -> int:
    """Генерирует случайное число заданной длины."""
    if length == 1:
        return random.randint(0, 9)
    return random.randint(10**(length-1), 10**length - 1)

def generate_addition_pool(size: int = 125000) -> list[tuple[int, int]]:
    pool = set()
    
    # Каскадные переносы: ~5% выборки
    edge_cases_target = int(size * 0.05)
    while len(pool) < edge_cases_target:
        base = random.randint(1, 99999)
        n_nines = random.randint(1, 4)
        base = (base // (10**n_nines)) * (10**n_nines) + (10**n_nines - 1)
        
        addend = random.randint(1, 1000)
        
        if base + addend <= 99999:
            pool.add((base, addend))
            pool.add((addend, base)) 

    # Стратифицированное сэмплирование
    while len(pool) < size:
        len_a = random.randint(1, 5)
        len_b = random.randint(1, 5)
        
        a = get_random_num(len_a)
        b = get_random_num(len_b)
        
        if a + b <= 99999:
            pool.add((a, b))
            
    pool_list = list(pool)
    random.shuffle(pool_list)
    return pool_list


def generate_subtraction_pool(size: int = 125000) -> list[tuple[int, int]]:
    pool = set()
    
    # Каскадные заемы: ~5% выборки
    edge_cases_target = int(size * 0.05)
    while len(pool) < edge_cases_target:
        base = random.randint(1, 99999)
        n_zeros = random.randint(1, 4)
        base = (base // (10**n_zeros)) * (10**n_zeros)
        
        if base == 0:
            base = 10**n_zeros
            
        subtrahend = random.randint(1, 1000)
        
        if base >= subtrahend:
            pool.add((base, subtrahend))

    # Стратифицированное сэмплирование
    while len(pool) < size:
        len_a = random.randint(1, 5)
        len_b = random.randint(1, 5)
        
        a = get_random_num(len_a)
        b = get_random_num(len_b)
        
        if a < b:
            a, b = b, a
            
        pool.add((a, b))
        
    pool_list = list(pool)
    random.shuffle(pool_list)
    return pool_list


In [6]:
 Генерируем пулы

addition_pairs = generate_addition_pool(125000)
subtraction_pairs = generate_subtraction_pool(125000)

print(f"Сгенерировано: сложений - {len(addition_pairs)}, вычитаний - {len(subtraction_pairs)}")

print("Примеры сложений:", addition_pairs[:5])
print("Примеры вычитаний:", subtraction_pairs[:5])

Сгенерировано: сложений - 125000, вычитаний - 125000
Примеры сложений: [(7174, 8090), (60, 4898), (42, 325), (415, 2), (97, 60684)]
Примеры вычитаний: [(845, 69), (726, 342), (326, 53), (82930, 42), (4424, 127)]


# 3 - Генерация train и test файлов

In [1]:
import json
import random
import os

# Train/Test Split и объединение
test_size_per_op = 5000

test_add = addition_pairs[:test_size_per_op]
train_add = addition_pairs[test_size_per_op:]

test_sub = subtraction_pairs[:test_size_per_op]
train_sub = subtraction_pairs[test_size_per_op:]

test_data = [(a, b, '+') for a, b in test_add] + [(a, b, '-') for a, b in test_sub]
train_data = [(a, b, '+') for a, b in train_add] + [(a, b, '-') for a, b in train_sub]

random.shuffle(train_data)

# Форматирование и подсчет максимальной длины
def format_prompt(a: int, b: int, op: str) -> str:
    a_str = " ".join(list(str(a).zfill(5)))
    b_str = " ".join(list(str(b).zfill(5)))
    return f"{a_str} {op} {b_str} ="

train_raw_records = []
max_tokens = 0

for a, b, op in train_data:
    prompt = format_prompt(a, b, op)
    target = generate_addition_cot(a, b) if op == '+' else generate_subtraction_cot(a, b)
    
    full_text = f"{prompt} {target}"
    n_tokens = len(full_text.split(" "))
    if n_tokens > max_tokens:
        max_tokens = n_tokens
        
    train_raw_records.append((prompt, target, full_text))

test_raw_records = []
for a, b, op in test_data:
    prompt = format_prompt(a, b, op)
    target = generate_addition_cot(a, b) if op == '+' else generate_subtraction_cot(a, b)
    
    full_text = f"{prompt} {target}"
    n_tokens = len(full_text.split(" "))
    if n_tokens > max_tokens:
        max_tokens = n_tokens
        
    test_raw_records.append((prompt, target, full_text))

# Паддинг и сохранение по нужному пути
def pad_string(text: str, current_len: int, target_len: int) -> str:
    pads_needed = target_len - current_len
    if pads_needed > 0:
        return text + " " + " ".join(["<pad>"] * pads_needed)
    return text

save_dir = os.path.join("..", "data")
os.makedirs(save_dir, exist_ok=True)

train_file_path = os.path.join(save_dir, "train.jsonl")
test_file_path = os.path.join(save_dir, "test.jsonl")

# Сохранение Train
with open(train_file_path, "w", encoding="utf-8") as f:
    for prompt, target, full_text in train_raw_records:
        current_len = len(full_text.split(" "))
        padded_text = pad_string(full_text, current_len, max_tokens)
        record = {"text": padded_text}
        f.write(json.dumps(record) + "\n")

# Сохранение Test
with open(test_file_path, "w", encoding="utf-8") as f:
    for prompt, target, full_text in test_raw_records:
        current_len = len(full_text.split(" "))
        padded_text = pad_string(full_text, current_len, max_tokens)
        record = {
            "prompt": prompt,
            "target": target,
            "text": padded_text
        }
        f.write(json.dumps(record) + "\n")

print(f"Максимальная длина в токенах: {max_tokens}")
print(f"Файлы сохранены в:\n- {os.path.abspath(train_file_path)}\n- {os.path.abspath(test_file_path)}\n")

# Вывод примеров из сохраненных данных
print("Примеры из train.jsonl")
with open(train_file_path, "r", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline().strip())

print("\nПримеры из test.jsonl")
with open(test_file_path, "r", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline().strip())

NameError: name 'addition_pairs' is not defined